# XLSum mT5-base Multilingual Notebook with Summary + Translation Demo

## 1. Install dependencies

In [ ]:
!pip -q uninstall -y datasets
!pip -q install "datasets==3.6.0" "transformers>=4.46,<5" evaluate sentencepiece accelerate rouge_score pandas


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 1.7 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 491.5/491.5 kB 11.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 35.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 20.5 MB/s eta 0:00:00


## 2. Imports

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch

from datasets import load_dataset, concatenate_datasets
import evaluate

from transformers import (
    AutoModelForSeq2SeqLM,
    AutoTokenizer,
    DataCollatorForSeq2Seq,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
)


## 3. Configuration

In [ ]:
MODEL_NAME = "google/mt5-base"

LANGUAGES = ["english", "spanish", "french", "russian", "portuguese"]

MAX_INPUT_LEN = 512
MAX_TARGET_LEN = 64

TRAIN_SAMPLES_PER_LANG = 1500
VAL_SAMPLES_PER_LANG = 200
TEST_SAMPLES_PER_LANG = 200

LEARNING_RATE = 1e-4
PER_DEVICE_TRAIN_BATCH_SIZE = 2
PER_DEVICE_EVAL_BATCH_SIZE = 2
GRADIENT_ACCUMULATION_STEPS = 4
NUM_TRAIN_EPOCHS = 3
NUM_BEAMS = 4

RUN_TAG = "_".join([lang[:2] for lang in LANGUAGES])
OUTPUT_DIR = f"./mt5_base_xlsum_multilingual_{RUN_TAG}"
PLOTS_DIR = os.path.join(OUTPUT_DIR, "report_plots")
TABLES_DIR = os.path.join(OUTPUT_DIR, "report_tables")

os.makedirs(PLOTS_DIR, exist_ok=True)
os.makedirs(TABLES_DIR, exist_ok=True)

print("HF model:", MODEL_NAME)
print("Languages:", LANGUAGES)


HF model: google/mt5-base
Languages: ['english', 'spanish', 'french', 'russian', 'portuguese']


## 4. Load XLSum dataset and build multilingual splits

In [ ]:
def add_lang_column(example, lang_name):
    example["lang_name"] = lang_name
    return example

train_parts, val_parts, test_parts = [], [], []
dataset_overview_rows = []

for lang_name in LANGUAGES:
    ds = load_dataset("csebuetnlp/xlsum", lang_name)

    dataset_overview_rows.append({
        "Language": lang_name,
        "Train Available": len(ds["train"]),
        "Validation Available": len(ds["validation"]),
        "Test Available": len(ds["test"]),
    })

    train_part = ds["train"].shuffle(seed=42).select(
        range(min(TRAIN_SAMPLES_PER_LANG, len(ds["train"])))
    ).map(lambda x, ln=lang_name: add_lang_column(x, ln))

    val_part = ds["validation"].shuffle(seed=42).select(
        range(min(VAL_SAMPLES_PER_LANG, len(ds["validation"])))
    ).map(lambda x, ln=lang_name: add_lang_column(x, ln))

    test_part = ds["test"].shuffle(seed=42).select(
        range(min(TEST_SAMPLES_PER_LANG, len(ds["test"])))
    ).map(lambda x, ln=lang_name: add_lang_column(x, ln))

    train_parts.append(train_part)
    val_parts.append(val_part)
    test_parts.append(test_part)

train_ds = concatenate_datasets(train_parts).shuffle(seed=42)
val_ds = concatenate_datasets(val_parts).shuffle(seed=42)
test_ds = concatenate_datasets(test_parts).shuffle(seed=42)

dataset_overview_df = pd.DataFrame(dataset_overview_rows)
dataset_overview_df


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

xlsum.py: 0.00B [00:00, ?B/s]

## 5. Inspect an example

In [ ]:
sample = train_ds[0]
print("Language:", sample["lang_name"])
print("Article preview:", sample["text"][:700], "...")
print("\nReference summary:", sample["summary"])

## 6. Confirm multilingual subset sizes

In [ ]:
split_summary = pd.DataFrame({
    "Split": ["Train", "Validation", "Test"],
    "Samples Used": [len(train_ds), len(val_ds), len(test_ds)]
})
split_summary

## 7. Dataset summary tables for the report

In [ ]:
language_sample_summary = pd.DataFrame([
    {
        "Language": lang,
        "Train Used": min(TRAIN_SAMPLES_PER_LANG, dataset_overview_df.loc[dataset_overview_df["Language"] == lang, "Train Available"].iloc[0]),
        "Validation Used": min(VAL_SAMPLES_PER_LANG, dataset_overview_df.loc[dataset_overview_df["Language"] == lang, "Validation Available"].iloc[0]),
        "Test Used": min(TEST_SAMPLES_PER_LANG, dataset_overview_df.loc[dataset_overview_df["Language"] == lang, "Test Available"].iloc[0]),
    }
    for lang in LANGUAGES
])

dataset_summary = split_summary.copy()
display(language_sample_summary)
display(dataset_summary)

In [ ]:
dataset_summary.to_csv(os.path.join(TABLES_DIR, "dataset_summary.csv"), index=False)
language_sample_summary.to_csv(os.path.join(TABLES_DIR, "language_sample_summary.csv"), index=False)
print("Saved tables to:", TABLES_DIR)

## 8. Visualize source and target length distributions

In [ ]:
def get_lengths(ds, text_col="text", summary_col="summary", max_rows=1000):
    rows = min(max_rows, len(ds))
    article_lens = [len(ds[i][text_col].split()) for i in range(rows)]
    summary_lens = [len(ds[i][summary_col].split()) for i in range(rows)]
    langs = [ds[i]["lang_name"] for i in range(rows)] if "lang_name" in ds.column_names else None
    return article_lens, summary_lens, langs

article_lens, summary_lens, sampled_langs = get_lengths(train_ds, max_rows=1000)

print("Avg article length:", round(np.mean(article_lens), 2))
print("Avg summary length:", round(np.mean(summary_lens), 2))

In [ ]:
plt.figure(figsize=(8, 5))
plt.hist(summary_lens, bins=30)
plt.xlabel("Summary Length (words)")
plt.ylabel("Count")
plt.title("Summary Length Distribution (Multilingual XLSum)")
plt.tight_layout()
plot_path = os.path.join(PLOTS_DIR, f"summary_length_hist_{RUN_TAG}.png")
plt.savefig(plot_path, dpi=200, bbox_inches="tight")
plt.show()

print("Saved:", plot_path)

## 9. Load tokenizer and model

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=False)
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)

if hasattr(model, "gradient_checkpointing_enable"):
    model.gradient_checkpointing_enable()

print("Tokenizer class:", tokenizer.__class__.__name__)
print("Model loaded:", MODEL_NAME)

## 10. Preprocessing

In [ ]:
prefix_template = "summarize {lang}: "

def preprocess_function(batch):
    prefixed_inputs = []
    for text, lang_name in zip(batch["text"], batch["lang_name"]):
        prefixed_inputs.append(prefix_template.format(lang=lang_name) + text)

    model_inputs = tokenizer(
        prefixed_inputs,
        max_length=MAX_INPUT_LEN,
        truncation=True,
        padding=False
    )

    labels = tokenizer(
        text_target=batch["summary"],
        max_length=MAX_TARGET_LEN,
        truncation=True,
        padding=False
    )

    model_inputs["labels"] = labels["input_ids"]
    return model_inputs


## 11. Tokenize datasets

In [ ]:
tokenized_train = train_ds.map(preprocess_function, batched=True, remove_columns=train_ds.column_names)
tokenized_val = val_ds.map(preprocess_function, batched=True, remove_columns=val_ds.column_names)
tokenized_test = test_ds.map(preprocess_function, batched=True, remove_columns=test_ds.column_names)

## 12. ROUGE metric

In [ ]:
rouge = evaluate.load("rouge")

def compute_metrics(eval_pred):
    predictions, labels = eval_pred

    if isinstance(predictions, tuple):
        predictions = predictions[0]

    predictions = np.where(
        (predictions < 0) | (predictions >= tokenizer.vocab_size),
        tokenizer.pad_token_id,
        predictions
    )

    decoded_preds = tokenizer.batch_decode(predictions, skip_special_tokens=True)

    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)

    decoded_preds = [pred.strip() for pred in decoded_preds]
    decoded_labels = [label.strip() for label in decoded_labels]

    result = rouge.compute(
        predictions=decoded_preds,
        references=decoded_labels,
        use_stemmer=False
    )

    return {
        "rouge1": round(result["rouge1"], 4),
        "rouge2": round(result["rouge2"], 4),
        "rougeL": round(result["rougeL"], 4),
        "rougeLsum": round(result["rougeLsum"], 4),
    }

## 13. Data collator

In [ ]:
data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=model,
    padding=True
)

## 14. Training arguments

In [ ]:
supports_bf16 = (
    torch.cuda.is_available()
    and torch.cuda.get_device_capability(0)[0] >= 8
)

training_args = Seq2SeqTrainingArguments(
    output_dir=OUTPUT_DIR,
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=LEARNING_RATE,
    per_device_train_batch_size=PER_DEVICE_TRAIN_BATCH_SIZE,
    per_device_eval_batch_size=PER_DEVICE_EVAL_BATCH_SIZE,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
    weight_decay=0.01,
    save_total_limit=2,
    num_train_epochs=NUM_TRAIN_EPOCHS,
    predict_with_generate=True,
    generation_max_length=MAX_TARGET_LEN,
    generation_num_beams=NUM_BEAMS,
    load_best_model_at_end=True,
    metric_for_best_model="eval_rougeLsum",
    greater_is_better=True,
    bf16=supports_bf16,
    fp16=torch.cuda.is_available() and not supports_bf16,
    logging_steps=25,
    report_to="none",
    logging_dir=os.path.join(OUTPUT_DIR, "logs"),
)

## 15. Trainer

In [ ]:
trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)


## 16. Train

In [ ]:
train_result = trainer.train()
train_result

## 17. Evaluate on test set

In [ ]:
test_results = trainer.evaluate(eval_dataset=tokenized_test, metric_key_prefix="test")
test_results

## 18. Save a clean results table for the report

In [ ]:
results_df = pd.DataFrame([{
    "Model": "mT5-base",
    "Dataset": "XLSum",
    "Training Setup": "Multilingual joint training",
    "Languages": ", ".join(LANGUAGES),
    "test_rouge1": test_results.get("test_rouge1"),
    "test_rouge2": test_results.get("test_rouge2"),
    "test_rougeL": test_results.get("test_rougeL"),
    "test_rougeLsum": test_results.get("test_rougeLsum"),
    "test_loss": test_results.get("test_loss"),
}])

results_df

In [ ]:
results_csv = os.path.join(TABLES_DIR, f"rouge_results_{RUN_TAG}.csv")
results_df.to_csv(results_csv, index=False)
print("Saved:", results_csv)

In [ ]:
rouge_plot_df = pd.DataFrame({
    "Metric": ["ROUGE-1", "ROUGE-2", "ROUGE-L", "ROUGE-Lsum"],
    "Score": [
        float(test_results.get("test_rouge1", 0)),
        float(test_results.get("test_rouge2", 0)),
        float(test_results.get("test_rougeL", 0)),
        float(test_results.get("test_rougeLsum", 0)),
    ]
})

plt.figure(figsize=(8, 5))
plt.bar(rouge_plot_df["Metric"], rouge_plot_df["Score"])
plt.xlabel("Metric")
plt.ylabel("Score")
plt.title("ROUGE Scores on Multilingual XLSum Test Set")
plt.tight_layout()

plot_path = os.path.join(PLOTS_DIR, f"rouge_bar_chart_{RUN_TAG}.png")
plt.savefig(plot_path, dpi=200, bbox_inches="tight")
plt.show()

print("Saved:", plot_path)
rouge_plot_df

In [ ]:
rouge_plot_df = pd.DataFrame({
    "Metric": ["ROUGE-1", "ROUGE-2", "ROUGE-L", "ROUGE-Lsum"],
    "Score": [
        float(test_results.get("test_rouge1", 0)),
        float(test_results.get("test_rouge2", 0)),
        float(test_results.get("test_rougeL", 0)),
        float(test_results.get("test_rougeLsum", 0)),
    ]
})

plt.figure(figsize=(8, 5))
plt.bar(rouge_plot_df["Metric"], rouge_plot_df["Score"])
plt.xlabel("Metric")
plt.ylabel("Score")
plt.title("ROUGE Scores on Multilingual XLSum Test Set")
plt.tight_layout()
plot_path = os.path.join(PLOTS_DIR, f"rouge_bar_chart_{RUN_TAG}.png")
plt.savefig(plot_path, dpi=200, bbox_inches="tight")
plt.show()

print("Saved:", plot_path)
rouge_plot_df

## 20. Training and evaluation loss curves

In [ ]:
log_history = trainer.state.log_history
logs_df = pd.DataFrame(log_history)
logs_df.head()

In [ ]:
train_loss_df = logs_df[logs_df["loss"].notna()].copy() if "loss" in logs_df.columns else pd.DataFrame()
eval_loss_df = logs_df[logs_df["eval_loss"].notna()].copy() if "eval_loss" in logs_df.columns else pd.DataFrame()

print("Train loss rows:", len(train_loss_df))
print("Eval loss rows:", len(eval_loss_df))

In [ ]:
if not train_loss_df.empty:
    plt.figure(figsize=(8, 5))
    plt.plot(train_loss_df["step"], train_loss_df["loss"], marker="o")
    plt.xlabel("Step")
    plt.ylabel("Training Loss")
    plt.title("Training Loss Curve (Multilingual XLSum)")
    plt.tight_layout()
    plot_path = os.path.join(PLOTS_DIR, f"training_loss_curve_{RUN_TAG}.png")
    plt.savefig(plot_path, dpi=200, bbox_inches="tight")
    plt.show()
    print("Saved:", plot_path)
else:
    print("No training loss logged.")

In [ ]:
if not eval_loss_df.empty:
    plt.figure(figsize=(8, 5))
    plt.plot(eval_loss_df["epoch"], eval_loss_df["eval_loss"], marker="o")
    plt.xlabel("Epoch")
    plt.ylabel("Validation Loss")
    plt.title("Validation Loss Curve (Multilingual XLSum)")
    plt.tight_layout()
    plot_path = os.path.join(PLOTS_DIR, f"eval_loss_curve_{RUN_TAG}.png")
    plt.savefig(plot_path, dpi=200, bbox_inches="tight")
    plt.show()
    print("Saved:", plot_path)
else:
    print("No eval loss logged.")

## 21. Generate summaries for qualitative examples

In [ ]:
def generate_summary(text, src_lang="english", max_new_tokens=64):
    if src_lang not in LANGUAGES:
        raise ValueError(f"src_lang must be one of {LANGUAGES}")

    device = model.device
    prompt = prefix_template.format(lang=src_lang) + text

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=MAX_INPUT_LEN
    )
    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            num_beams=NUM_BEAMS
        )

    return tokenizer.decode(outputs[0], skip_special_tokens=True)

In [ ]:
example_rows = []

sample_indices = list(range(min(5, len(test_ds))))
for idx in sample_indices:
    row = test_ds[idx]
    pred = generate_summary(row["text"], src_lang=row["lang_name"], max_new_tokens=MAX_TARGET_LEN)
    example_rows.append({
        "language": row["lang_name"],
        "reference_summary": row["summary"],
        "generated_summary": pred,
    })

examples_df = pd.DataFrame(example_rows)
examples_df.head()

In [ ]:
examples_csv = os.path.join(TABLES_DIR, "qualitative_examples_multilingual_xlsum.csv")
examples_df.to_csv(examples_csv, index=False)
print("Saved:", examples_csv)

## 22. Save model and tokenizer

In [ ]:
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f"Saved model and tokenizer to: {OUTPUT_DIR}")

In [ ]:
inference_device = "cuda" if torch.cuda.is_available() else "cpu"

inference_model = AutoModelForSeq2SeqLM.from_pretrained(OUTPUT_DIR).to(inference_device)
inference_tokenizer = AutoTokenizer.from_pretrained(OUTPUT_DIR)

print("Inference model loaded from:", OUTPUT_DIR)
print("Inference device:", inference_device)

In [ ]:
def generate_summary_api(text, src_lang, tgt_lang=None, max_new_tokens=64, num_beams=4):
    if src_lang not in LANGUAGES:
        raise ValueError(f"src_lang must be one of {LANGUAGES}")

    if tgt_lang is None:
        tgt_lang = src_lang

    if tgt_lang not in LANGUAGES:
        raise ValueError(
            "This mT5 notebook is trained for same-language summarization only. "
            f"tgt_lang must be one of {LANGUAGES}."
        )

    if tgt_lang != src_lang:
        raise ValueError(
            "Direct cross-lingual summarization is not supervised in this notebook. "
            "Use summarize_then_translate(...) instead."
        )

    prompt = prefix_template.format(lang=src_lang) + text
    inputs = inference_tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=1024
    )
    inputs = {k: v.to(inference_device) for k, v in inputs.items()}

    with torch.no_grad():
        output_ids = inference_model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            num_beams=num_beams
        )

    return inference_tokenizer.decode(output_ids[0], skip_special_tokens=True)

## 23. Add translation model for summary-to-target-language demo

In [ ]:
TRANSLATOR_MODEL_NAME = "facebook/nllb-200-distilled-600M"

translator_tokenizer = AutoTokenizer.from_pretrained(TRANSLATOR_MODEL_NAME)
translator_model = AutoModelForSeq2SeqLM.from_pretrained(TRANSLATOR_MODEL_NAME).to(inference_device)
translator_model.eval()

NLLB_LANG_CODES = {
    "english": "eng_Latn",
    "spanish": "spa_Latn",
    "french": "fra_Latn",
    "russian": "rus_Cyrl",
    "portuguese": "por_Latn",
}

print("Translator model loaded:", TRANSLATOR_MODEL_NAME)
print("Supported translation languages:", list(NLLB_LANG_CODES.keys()))

In [ ]:
def translate_text(text, src_lang, tgt_lang, max_new_tokens=128, num_beams=4):
    if src_lang not in NLLB_LANG_CODES:
        raise ValueError(f"src_lang must be one of {list(NLLB_LANG_CODES.keys())}")
    if tgt_lang not in NLLB_LANG_CODES:
        raise ValueError(f"tgt_lang must be one of {list(NLLB_LANG_CODES.keys())}")

    src_code = NLLB_LANG_CODES[src_lang]
    tgt_code = NLLB_LANG_CODES[tgt_lang]

    translator_tokenizer.src_lang = src_code
    inputs = translator_tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        max_length=512
    )
    inputs = {k: v.to(inference_device) for k, v in inputs.items()}

    with torch.no_grad():
        output_ids = translator_model.generate(
            **inputs,
            forced_bos_token_id=translator_tokenizer.convert_tokens_to_ids(tgt_code),
            max_new_tokens=max_new_tokens,
            num_beams=num_beams
        )

    return translator_tokenizer.decode(output_ids[0], skip_special_tokens=True)


def summarize_then_translate(text, src_lang, tgt_lang, summary_max_new_tokens=64, translation_max_new_tokens=128, num_beams=4):
    source_summary = generate_summary_api(
        text=text,
        src_lang=src_lang,
        tgt_lang=src_lang,
        max_new_tokens=summary_max_new_tokens,
        num_beams=num_beams
    )

    if tgt_lang == src_lang:
        translated_summary = source_summary
    else:
        translated_summary = translate_text(
            text=source_summary,
            src_lang=src_lang,
            tgt_lang=tgt_lang,
            max_new_tokens=translation_max_new_tokens,
            num_beams=num_beams
        )

    return {
        "source_summary": source_summary,
        "translated_summary": translated_summary,
    }

In [ ]:
# Demo: summarize in the source language, then translate the summary
demo_example = test_ds[0]
demo_target_lang = "english" if demo_example["lang_name"] != "english" else "portuguese"

demo_result = summarize_then_translate(
    text=demo_example["text"],
    src_lang=demo_example["lang_name"],
    tgt_lang=demo_target_lang
)

print("Source language:", demo_example["lang_name"])
print("Target language:", demo_target_lang)
print()
print("Source-language summary:")
print(demo_result["source_summary"])
print()
print("Translated summary:")
print(demo_result["translated_summary"])

In [ ]:
# Replace these values with custom article and desired languages

manual_text = """"Na Fran√ßa, o Le Monde diz que a Uni√£o Europ√©ia ter√° que fazer uma escolha que ela gostaria de evitar: "fortalecer suas rela√ß√µes com a R√∫ssia ou apoiar a democracia na Ucr√¢nia". Com o texto intitulado "Salvar a Ucr√¢nia", o jornal franc√™s diz que os europeus "est√£o tendo dificuldades em esconder o seu constragimento". O El Pa√≠s, da Espanha, afirma que "a poss√≠vel desestabiliza√ß√£o da Ucr√¢nia pode comprometer para sempre a rela√ß√£o, que se deteriora cada vez mais, entre a R√∫ssia e a Uni√£o Europ√©ia". "A solu√ß√£o pac√≠fica para a crise tem que incluir o reconhecimento de que n√£o se pode impor um presidente eleito de maneira fraudulenta a um pa√≠s", diz o jornal. R√∫ssia x EUA Na Alemanha, o Frankfurter Rundschau afirma que tanto a R√∫ssia quanto os Estados Unidos est√£o tentando usar a divis√£o na Ucr√¢nia em seu pr√≥prio benef√≠cio. Segundo o di√°rio, o interesse dos Estados Unidos na Ucr√¢nia deve ser visto como uma tentativa de "limitar a influ√™ncia da R√∫ssia". Na Rep√∫blica Checa, o jornal Pravo diz que Vladimir Putin n√£o est√° √† vontade com o apoio estrangeiro √† oposi√ß√£o ucraniana. De acordo com o Pravo, Putin v√™ esse apoio "como mais uma tentativa de frear a R√∫ssia, especialmente depois que os pa√≠ses b√°lticos se juntaram √† Otan". Na Ucr√¢nia, o jornal Den, pr√≥-governo, afirma que "o pr√≥prio fato de que, apesar da tens√£o social, nenhuma gota de sangue foi derramada √© uma prova de que aconteceu uma revolu√ß√£o na Ucr√¢nia". "√â uma revolu√ß√£o nas mentes das pessoas, que perceberam que a democracia est√° em suas m√£os", afirma. O Ukrayina Moloda, que √© de oposi√ß√£o, diz que como o parlamento n√£o conseguiu tomar nenhuma decis√£o at√© agora, "a √∫nica fonte de poder na Ucr√¢nia √© a sua popula√ß√£o". Maturidade Na R√∫ssia, o Nezavisimaya Gazeta diz que a sociedade ucraniana "√© muito mais madura do que a sua vizinha R√∫ssia". "Uma das raz√µes √© que os ucranianos conseguem o que os russos n√£o conseguem: combinar nacionalismo com aspira√ß√µes √† liberdade". "Parab√©ns, ucranianos. N√≥s n√£o temos nada a ensinar a voc√™s", diz o jornal. O Novaya Gazeta, tamb√©m russo, diz: "Quem quer que seja o vencedor na Ucr√¢nia, a R√∫ssia j√° perdeu". Brasil O jornal Financial Times, da Gr√£-Bretanha, traz uma reportagem em que detalha os planos do presidente Luiz In√°cio Lula da Silva de construir um sistema de canais de irriga√ß√£o no sert√£o nordestino "para terminar com uma das maiores ondas de migra√ß√£o do mundo". O Financial Times conta que, meio s√©culo atr√°s, o pr√≥prio Lula "abandonou a regi√£o que parece um deserto numa viagem √©pica de 13 dias para S√£o Paulo, na carroceria de um caminh√£o". Segundo o jornal, o presidente defende o custo da obra, cerca de R$ 5 bilh√µes, afirmando que custaria mais n√£o realizar o projeto. Mas o Financial Times afirma que h√° riscos para a iniciativa: "N√£o h√° garantias de que o sucessor de Lula completar√° o projeto, condenando-o ao mesmo destino de outros elefantes brancos abandonados na regi√£o. A constru√ß√£o dos dutos que trariam a √°gua a pequenas cidades depende largamente de governos estaduais, que podem n√£o ter dinheiro ou vontade de fazer isso, particularmente naqueles Estados governados por partidos de oposi√ß√£o". Na Fran√ßa, o Le Monde traz uma reportagem sobre o Brasil em que come√ßa falando sobre um ataque s√°bado a um acampamento de sem-teto no Vale do Jequitinhonha, em Minas Gerais, que deixou cinco mortos e 13 feridos. Em seguida, o jornal passa a falar da "prefeita em fim de mandato de S√£o Paulo, Marta Suplicy, que atribui o fracasso de sua reelei√ß√£o √† pol√≠tica econ√¥mica". O Le Monde fala em mal-estar no governo e cita a sa√≠da de v√°rios integrantes dos altos escal√µes nas √∫ltimas semanas como um sinal de crise. O di√°rio franc√™s conclui a reportagem sobre o Brasil falando da crise gerada pela decis√£o do prefeito do Rio, C√©sar Maia, de se candidatar √† presid√™ncia em 2006. E termina dizendo: "Nas cidades, as prioridades da opini√£o p√∫blica n√£o s√£o a terra nem a fome, mas o emprego e a seguran√ßa"."""
manual_src_lang = "portuguese"
manual_tgt_lang = "english"

manual_result = summarize_then_translate(
    text=manual_text,
    src_lang=manual_src_lang,
    tgt_lang=manual_tgt_lang
)

print("Manual source language:", manual_src_lang)
print("Manual target language:", manual_tgt_lang)
print()
print("Source-language summary:")
print(manual_result["source_summary"])
print()
print("Translated summary:")
print(manual_result["translated_summary"])